In [ ]:
import os
import cv2
import numpy as np

def process_image(image):
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Green color range
    lower_bound = np.array([35, 40, 40])
    upper_bound = np.array([85, 255, 255])
    mask = cv2.inRange(hsv_image, lower_bound, upper_bound)

    # Clean the mask
    kernel = np.ones((5, 5), np.uint8)
    mask_cleaned = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask_cleaned = cv2.morphologyEx(mask_cleaned, cv2.MORPH_OPEN, kernel)

    # Find objects
    contours, _ = cv2.findContours(
        mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    h, w = image.shape[:2]
    center_x, center_y = w // 2, h // 2

    # Define a central region to select objects close to the center
    central_region = (
        int(w * 0.25),
        int(h * 0.25),
        int(w * 0.5),
        int(h * 0.5)
    )  # (x, y, width, height)

    # Create a new mask for the plant
    mask_final = np.zeros_like(mask_cleaned)

    for cnt in contours:
        x, y, cw, ch = cv2.boundingRect(cnt)

        # Check whether the bounding rectangle overlaps with the central region
        if (
            x + cw > central_region[0]
            and x < central_region[0] + central_region[2]
            and y + ch > central_region[1]
            and y < central_region[1] + central_region[3]
        ):
            cv2.drawContours(
                mask_final, [cnt], -1, 255, thickness=cv2.FILLED
            )

    # Apply the mask
    result = cv2.bitwise_and(image, image, mask=mask_final)
    result[mask_final == 0] = [255, 255, 255]

    return result, mask_final


# Define input and output paths
input_root = r"D:\all\AI\photo\xxxxx"          # Change this to your original folder
output_root = r"D:\all\AI\photo\xxxxxout333"   # Change this to your output folder

# Traverse all files and subfolders
for root, dirs, files in os.walk(input_root):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            input_path = os.path.join(root, file)
            rel_path = os.path.relpath(input_path, input_root)
            output_path = os.path.join(output_root, rel_path)

            # Read the image
            image = cv2.imread(input_path)

            if image is None:
                print(f"Failed to read image: {input_path}")
                continue

            # Process the image
            processed, _ = process_image(image)

            # Create the output directory if it does not exist
            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            # Save the processed image
            cv2.imwrite(output_path, processed)
            print(f"Processed image saved: {output_path}")